In [10]:
import h5py  # before icecube
from icecube import dataio, icetray
from pathlib import Path
import re
import pandas as pd

In [11]:
JOB_LOG = Path("/home/kbas/scratch/String340MC/Logs/job_38826931_Muon_full_geometry.log")
LOGDIR  = Path("/home/kbas/scratch/String340MC/Logs/Muon_pmt_response_Full_Geometry")
OUTDIR  = Path("/home/kbas/scratch/String340MC/Full_Geometry/Muon_PMT_Response")

FLAVOR   = "muon"      # per-file log prefix
PULSEMAP = "EventPulseSeries_nonoise"

## 1. Job log'undaki [ok] satırlarını parse et

In [12]:
pat_ok = re.compile(r"\[\d{2}:\d{2}:\d{2}\]\s+\[ok\]\s+(\S+)\s+\(([\d.]+)s\)")

entries = []
for line in JOB_LOG.read_text(errors="replace").splitlines():
    m = pat_ok.search(line)
    if m:
        i3name  = m.group(1)          # e.g. cls_1023.i3
        elapsed = float(m.group(2))
        stem    = i3name.replace(".i3", "")   # cls_1023
        entries.append({
            "i3name":  i3name,
            "stem":    stem,
            "elapsed": elapsed,
            "log":     LOGDIR / f"{FLAVOR}_{stem}.out",
            "gz":      OUTDIR / f"{FLAVOR}_{stem}.i3.gz",
        })

print(f"[ok] satır sayısı: {len(entries)}")
for e in entries[:5]:
    print(f"  {e['i3name']:20s}  elapsed={e['elapsed']}s  log_exists={e['log'].exists()}  gz_exists={e['gz'].exists()}")

[ok] satır sayısı: 311
  cls_1023.i3           elapsed=3.6s  log_exists=True  gz_exists=True
  cls_126.i3            elapsed=3.7s  log_exists=True  gz_exists=True
  cls_038.i3            elapsed=3.7s  log_exists=True  gz_exists=True
  cls_1575.i3           elapsed=3.7s  log_exists=True  gz_exists=True
  cls_1149.i3           elapsed=3.7s  log_exists=True  gz_exists=True


## 2. Per-file log içeriklerine bak

In [19]:
pat_success = re.compile(r"=== SUCCESS\s+elapsed=([\d.]+)s")
pat_summary = re.compile(r"\[.+?\]\s+kept=(\d+)\s+noise_only=(\d+)\s+absent_pulsemap=(\d+)\s+corrupt_frames=(\d+)")

log_rows = []
for e in entries:
    row = {"stem": e["stem"], "elapsed_job": e["elapsed"]}
    if not e["log"].exists():
        row["status"] = "LOG MISSING"
    else:
        text = e["log"].read_text(errors="replace")
        row["status"]  = "success" if pat_success.search(text) else "NO SUCCESS"
        m = pat_summary.search(text)
        if m:
            row["kept"]            = int(m.group(1))
            row["noise_only"]      = int(m.group(2))
            row["absent_pulsemap"] = int(m.group(3))
            row["corrupt_frames"]  = int(m.group(4))
    log_rows.append(row)

df_logs = pd.DataFrame(log_rows)
print(df_logs["status"].value_counts())
df_logs

status
success    311
Name: count, dtype: int64


,stem,elapsed_job,status
0,cls_1023,3.6,success
1,cls_126,3.7,success
2,cls_038,3.7,success
3,cls_1575,3.7,success
4,cls_1149,3.7,success
...,...,...,...
306,cls_9995,3.1,success
307,cls_9996,3.1,success
308,cls_9997,3.1,success
309,cls_9998,3.1,success


In [14]:
# kept=0 olanlar var mi?
if "kept" in df_logs.columns:
    print("kept istatistikleri:")
    print(df_logs["kept"].describe())
    print(f"\nkept=0: {(df_logs['kept'] == 0).sum()} dosya")
    print(df_logs[df_logs["kept"] == 0][["stem", "kept", "noise_only", "absent_pulsemap"]])

## 3. gz dosyalarına bak (log'da görünenler)

In [15]:
gz_rows = []
for e in entries:
    row = {"stem": e["stem"]}
    if not e["gz"].exists():
        row["gz_exists"] = False
        gz_rows.append(row)
        continue
    row["gz_exists"] = True
    row["size_mb"]   = e["gz"].stat().st_size / 1e6
    
    f = dataio.I3File(str(e["gz"]), "r")
    n_daq = 0
    total_hits = 0
    while f.more():
        try:
            frame = f.pop_frame()
        except Exception:
            continue
        if frame.Stop != icetray.I3Frame.DAQ:
            continue
        n_daq += 1
        if PULSEMAP in frame:
            try:
                total_hits += sum(len(v) for v in frame[PULSEMAP].values())
            except Exception:
                pass
    row["n_daq"]      = n_daq
    row["total_hits"] = total_hits
    gz_rows.append(row)
    print(f"{e['stem']:20s}  frames={n_daq:4d}  hits={total_hits:6d}  {row['size_mb']:.1f} MB")

df_gz = pd.DataFrame(gz_rows)

cls_1023              frames=   0  hits=     0  0.0 MB
cls_126               frames=   0  hits=     0  0.0 MB
cls_038               frames=   0  hits=     0  0.0 MB
cls_1575              frames=   0  hits=     0  0.0 MB
cls_1149              frames=   0  hits=     0  0.0 MB
cls_1243              frames=   0  hits=     0  0.0 MB
cls_1362              frames=   0  hits=     0  0.0 MB
cls_1347              frames=   0  hits=     0  0.0 MB
cls_1089              frames=   0  hits=     0  0.0 MB
cls_1337              frames=   0  hits=     0  0.0 MB
cls_1660              frames=   0  hits=     0  0.0 MB
cls_1455              frames=   0  hits=     0  0.0 MB
cls_1036              frames=   0  hits=     0  0.0 MB
cls_1208              frames=   0  hits=     0  0.0 MB
cls_092               frames=   0  hits=     0  0.0 MB
cls_1683              frames=   0  hits=     0  0.0 MB
cls_1836              frames=   0  hits=     0  0.0 MB
cls_195               frames=   0  hits=     0  0.0 MB
cls_1988  

## 4. Log + gz birleşik tablo

In [16]:
df = df_logs.merge(df_gz, on="stem", how="left")
df

,stem,elapsed_job,status,gz_exists,size_mb,n_daq,total_hits
0,cls_1023,3.6,success,True,0.00002,0,0
1,cls_126,3.7,success,True,0.00002,0,0
2,cls_038,3.7,success,True,0.00002,0,0
3,cls_1575,3.7,success,True,0.00002,0,0
4,cls_1149,3.7,success,True,0.00002,0,0
...,...,...,...,...,...,...,...
306,cls_9995,3.1,success,True,0.00002,0,0
307,cls_9996,3.1,success,True,0.00002,0,0
308,cls_9997,3.1,success,True,0.00002,0,0
309,cls_9998,3.1,success,True,0.00002,0,0


In [17]:
print("gz ozet:")
print(df[["n_daq", "total_hits", "size_mb"]].describe())

gz ozet:
       n_daq  total_hits    size_mb
count  311.0       311.0  311.00000
mean     0.0         0.0    0.00002
std      0.0         0.0    0.00000
min      0.0         0.0    0.00002
25%      0.0         0.0    0.00002
50%      0.0         0.0    0.00002
75%      0.0         0.0    0.00002
max      0.0         0.0    0.00002


In [24]:
from icecube import dataio

path = "/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_1023.i3"

f = dataio.I3File(path)

n_ok = 0
n_err = 0
max_consecutive_errors = 10
consecutive_errors = 0

while f.more():
    try:
        frame = f.pop_frame()
        n_ok += 1
        consecutive_errors = 0

        # burada frame ile ne yapacaksan yap
        # print(frame)

    except Exception as e:
        n_err += 1
        consecutive_errors += 1
        print(f"[ERR] attempted frame {n_ok + n_err}: {e}")

        # Eğer stream toparlanamıyorsa sonsuz döngüye girmemek için dur
        if consecutive_errors >= max_consecutive_errors:
            print("Üst üste çok hata geldi; stream muhtemelen toparlanamıyor.")
            break

print(f"sağlam={n_ok}  hatalı={n_err}")

[ERR] attempted frame 185: input stream error
sağlam=184  hatalı=1


In [27]:
from icecube import dataio

path = "/project/6008051/pone_simulation/MC000002-nu_mu-2_7-LeptonInjector-PROPOSAL-clsim-v10/Photon/cls_1023.i3"

f = dataio.I3File(path)

n = 0

try:
    while f.more():
        f.pop_frame()
        n += 1

    print(f"Toplam frame sayısı: {n}")
    print("Dosya tamamen okunabildi.")

except Exception as e:
    print(f"Okunabilen frame sayısı: {n}")
    print(f"Hata: {e}")
    print("Bu dosya tam okunamıyor; frame sayısı güvenilir toplam değil, sadece hata öncesi sayı.")

Okunabilen frame sayısı: 184
Hata: input stream error
Bu dosya tam okunamıyor; frame sayısı güvenilir toplam değil, sadece hata öncesi sayı.
